This notebook implements Bayesian hyperparameter optimization for Convolutional Neural Networks.

## 0. Mount drive

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries, get dataset

In [ ]:
import os
import random
import pickle
from google.colab import files

import scipy
import scipy.stats
import pandas as pd
import numpy as np

import hyperopt.fmin as hypfmin
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

!pip install pyyaml h5py

import sys
sys.path.insert(0, drive_dir)
import utils, cnn_conv2d, training_schemes

#load .csv into a DataFrame
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}Ecoli_data.csv', index_col= 0).dropna().reset_index(drop = True)

## 2. Load saved split of working dataset plus static test set.

In [ ]:
iter_ID   = 1 #choose an integer in [1,5] to load different sets
get_saved = False #set to *True* to use saved working/heldout sets; set to *False* to generate new sets

###
iteration = f'iteration_{iter_ID}/'
series_dict = utils.split_seeds(raw_data)

working_set, heldout_set = utils.get_split(series_dict = series_dict, 
                                           series_list = [sr for sr in range(1,57)],
                                           load_saved = get_saved,
                                           train_size = 0.9, #change to get different dataset ratios
                                           path = f'{drive_dir}_saved/{iteration}')

print(f'Working set contains {working_set.shape[0]} sequences for train and validation\nHeldout contains {heldout_set.shape[0]} sequences for testing')

## 3. Define CNN hyperparameters search space

In [ ]:
pheno = 'Protein'  
train_sz = 0.75
stratify = True #change to false if you want to use a standard, non-stratified splitting strategy with sklearn

batch = 64
batch_norm = False #change to *True* if you want to seach for achitectures that use batch normalization
learning_rate = 1e-3  
epochs = 100          
patience = 15       
evals = 30

model_config = dict({'train_size' : train_sz,
                     'phenotype' : pheno,
                     'stratify' : stratify
                    })

hyperparams = {'conv_width' : [8, 13, 15],
               'conv_filters' : [64, 128, 256],
               'conv_layers' : [2, 3, 4, 5],
               'dense_layers' : [2, 3, 4, 5],
               'conv_dropout' : [0.15, 0.2],
               'dense_dropout' : [0.1, 0.25, 0.5],
               'dense_units' : [64, 128, 256]}

space = {   'conv_width': hp.choice('conv_width', [8, 13, 15]),
            'conv_filters': hp.choice('conv_filters', [64, 128, 256]),
            'conv_layers': hp.choice('conv_layers', [2, 3, 4, 5]),
            'dense_layers': hp.choice('dense_layers', [2, 3, 4, 5]),
            'conv_dropout': hp.choice('conv_dropout',  [0.15, 0.2]),
            'dense_dropout': hp.choice('dense_dropout', [0.1, 0.25, 0.5]),
            'dense_units': hp.choice('dense_units', [64, 128, 256]),
        }

### 3.1 Function to handle the hyperparameter optimization

In [ ]:
def f_nn(params, 
         pheno:str = 'Protein',
         conv_tp:str = 'conv2D',
         batch:int = 64,
         batch_norm:bool = False,
         epochs:int = 100,
         patience:int = 15,
         learning_rate:float = 1e-3
         ):

  # MODEL ZONE
  model = cnn_conv2d.create_model(params, 
                                  learning_rate, 
                                  include_BN=batch_norm)

  history = History()
  earlystop = EarlyStopping(monitor = 'val_loss', 
                            mode = 'min', 
                            verbose = 1, 
                            patience = patience)
  modelcheckpoint = ModelCheckpoint(filepath = checkpoint_path,
                                    monitor = 'val_loss',
                                    verbose = 0,
                                    save_best_only = True,
                                    mode = 'auto')
  
  # keep track of where we are while the code in this cell is running
  global n
  print("\n", n)
  n+=1
  print(params)

  model.fit(X_train,
            y_train,
            validation_data = (X_val, y_val),
            callbacks = [history,
                         modelcheckpoint,
                         earlystop],
            verbose = 0,
            shuffle = True,
            batch_size = batch,
            epochs = epochs)
  
  print('MSE:', earlystop.best)

  return {'loss': earlystop.best, 'status': STATUS_OK}

## 4. Hyperparameter Optimization

**Note**, you may encounter the following error when using *training_schemes.split_sets* for the validation set:
```
ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
```
This occurs because of the binning process in the *verstack* package, and the small number of samples randomply sampled for each bin. 

To overcome the error, simply re-execute the code block (~3-5 times).


In [ ]:
from random import sample

!mkdir -p training_entire
checkpoint_path = "training_entire/cp-{epoch:04d}.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

import tensorflow as tf
from sklearn.metrics import mean_squared_error, r2_score
from keras.callbacks import History, EarlyStopping, ModelCheckpoint

cfg = tf.compat.v1.ConfigProto()
cfg.gpu_options.allow_growth = True
with tf.compat.v1.Session(config=cfg) as sess:

  series_list = [itr for itr in range(1,57)]

  _, temp_ = training_schemes.split_sets(series_dict = utils.split_seeds(working_set),
                                         series_list = series_list,
                                         model_config = model_config,
                                         cumul = True
                                        )

  model_config['train_size'] = 0.44
  # split temporary df to get a validation set
  train_, val_ = training_schemes.split_sets(series_dict = utils.split_seeds(temp_['cumul']),
                                             series_list = series_list,
                                             model_config = model_config,
                                             cumul = True
                                             )

  # one-hot encode resulting TVT dataframes
  # also split (OH(seqs), values) pairs
  X_train, y_train = cnn_conv2d.one_hot_encoding(train_['cumul'], 'Protein')
  X_val, y_val = cnn_conv2d.one_hot_encoding(val_['cumul'], 'Protein')

  n = 0
  trials = Trials()
  best = hypfmin(f_nn, space,
                algo = tpe.suggest,
                max_evals = evals,
                trials = trials
                )
  print('best: ')
  print(best)

  opt_params = {p:hyperparams[p][best[p]] for p in best}
  #opt_params = {}
  #for p in best:
    #opt_params[p] = hyperparams[p][best[p]]
    #opt_params
  print('\n Hyperparam search DONE (Y)!')
          
  with open(f'{drive_dir[:-10]}hyperOpt/opt_params_{ver}.pkl', 'wb') as q:
    pickle.dump(opt_params, q)
          
  print('\n Saved opt_params dict.')